In [26]:
!git clone https://github.com/sharul-ayub/malaysia-bank-employee-sentiment-analysis.git
%cd malaysia-bank-employee-sentiment-analysis

Cloning into 'malaysia-bank-employee-sentiment-analysis'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 69 (delta 25), reused 28 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (69/69), 419.43 KiB | 1.61 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/content/malaysia-bank-employee-sentiment-analysis


# 02 — AI-Assisted Sentiment Labelling and Manual Check

This notebook:

1. Loads the sentence-level dataset produced from the previous preprocessing step.
2. Uses `facebook/bart-large-mnli` for zero-shot sentiment labelling.
3. Records prediction confidence and flags low-confidence predictions for additional attention.
4. Exports the AI-labelled sentence-level dataset for manual review.
5. Manually reviews and corrects:

   * **sentence boundaries/splits**, and
   * **sentiment labels**.
6. Loads the manually reviewed and corrected dataset.
7. Converts the final validated sentiment labels to binary **Positive/Negative** labels.

The zero-shot model is used as **annotation assistance**, not as ground truth. Both the sentence splits and sentiment labels are manually reviewed before the dataset is treated as final.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(".")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
LABELED_DIR = PROJECT_ROOT / "data" / "labeled"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
LABELED_DIR.mkdir(parents=True, exist_ok=True)

sentence_path = PROCESSED_DIR / "01_sentence_level.csv"

sentence_df = pd.read_csv(sentence_path)

print("Loaded sentence-level dataset:", sentence_path)

display(sentence_df.head())
print("Rows:", len(sentence_df))

Loaded sentence-level dataset: data/processed/01_sentence_level.csv


,review_row_id,id,company_name,text_content,sentence_id,sentence_text,original_text
0,1,1,Cimb-Group,review_title,1,Clear object,Clear object
1,2,1,Cimb-Group,review_body_text,1,Good company and exposure for junior level.,Good company and exposure for junior level. Pr...
2,2,1,Cimb-Group,review_body_text,2,Promotion can be very fast to some work functi...,Good company and exposure for junior level. Pr...
3,3,2,HSBC,review_title,1,best place to work,best place to work
4,4,2,HSBC,review_body_text,1,"work life balance, good elearning, internaltio...","work life balance, good elearning, internaltio..."


Rows: 1416


## Install and load the zero-shot model

In [ ]:
%pip install -q transformers torch tqdm

In [ ]:
from tqdm.auto import tqdm
from transformers import pipeline

tqdm.pandas()

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

candidate_labels = [
    "Positive",
    "Negative",
    "Neutral"
]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

## Define AI labelling function

In [ ]:
def ai_label_text(text):
    text = str(text).strip()

    if text == "":
        return pd.Series(["Neutral", 0.0])

    result = classifier(
        text,
        candidate_labels,
        hypothesis_template="This employee review expresses {} sentiment."
    )

    return pd.Series([
        result["labels"][0],
        result["scores"][0]
    ])

In [ ]:
text_df = sentence_df.copy()

text_df[["AI_Label", "AI_Confidence"]] = (
    text_df["sentence_text"]
    .progress_apply(ai_label_text)
)

text_df["Review_Status"] = text_df["AI_Confidence"].apply(
    lambda score: "Manual Review" if score < 0.75 else "Accepted"
)

text_df["Final_Label"] = text_df["AI_Label"]

display(
    text_df[
        ["sentence_text", "AI_Label", "AI_Confidence",
         "Review_Status", "Final_Label"]
    ].head(20)
)

  0%|          | 0/1416 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,sentence_text,AI_Label,AI_Confidence,Review_Status,Final_Label
0,Clear object,Positive,0.479954,Manual Review,Positive
1,Good company and exposure for junior level.,Positive,0.949811,Accepted,Positive
2,Promotion can be very fast to some work functi...,Positive,0.945431,Accepted,Positive
3,best place to work,Positive,0.987800,Accepted,Positive
4,"work life balance, good elearning, internaltio...",Positive,0.978488,Accepted,Positive
5,some management are suck but you might meet so...,Negative,0.844942,Accepted,Negative
6,"work life balance, good elearning",Positive,0.957867,Accepted,Positive
7,low increament,Negative,0.656295,Manual Review,Negative
8,Productive workplace,Positive,0.962599,Accepted,Positive
9,"brilliant training initiatives, cool projects ...",Negative,0.928187,Accepted,Negative


## Export AI-labelled data for manual review

Export the AI-labelled sentence-level dataset for manual validation.

Manually review each record for:

1. **Sentence split correctness** — check whether each sentence has been split at the correct boundary. Correct, merge, or adjust incorrectly split sentences where necessary.
2. **Sentiment label correctness** — check whether the AI-generated sentiment label accurately represents the meaning of the sentence.

Pay particular attention to rows where `Review_Status = Manual Review`, while also correcting any high-confidence predictions that are semantically incorrect.

After reviewing both the sentence splits and sentiment labels on `data/labeled/02_ai_labeled_reviews_manual_checked.csv`, save the manually checked dataset.


In [ ]:
ai_output = LABELED_DIR / "01_ai_labeled_reviews.csv"
manual_checked_output = LABELED_DIR / "02_ai_labeled_reviews_manual_checked.csv"

text_df.to_csv(ai_output, index=False, encoding="utf-8-sig")
text_df.to_csv(manual_checked_output, index=False, encoding="utf-8-sig")

print("AI-labelled file saved:", ai_output)
print("After manual checking, save as:")
print(manual_checked_output)

AI-labelled file saved: data/labeled/01_ai_labeled_reviews.csv
After manual checking, save as:
data/labeled/02_ai_labeled_reviews_manual_checked.csv


## Load manually checked labels

In [27]:
if not manual_checked_output.exists():
    raise FileNotFoundError(
        f"Manual checked file not found: {manual_checked_output}\n"
        "Open 01_ai_labeled_reviews.csv, review Final_Label, then save the checked file."
    )

checked_df = pd.read_csv(manual_checked_output)

print("Loaded:", manual_checked_output)
display(checked_df.head())
print(checked_df["Final_Label"].value_counts(dropna=False))

Loaded: data/labeled/02_ai_labeled_reviews_manual_checked.csv


,AI_Label,Final_Label,AI_Confidence,Review_Status,sentence_text
0,Positive,3,0.354210,Manual Review,Traditional type of work experience.
1,Negative,3,0.354304,Manual Review,Place to look for experience only (1 - 2 years...
2,Neutral,2,0.357208,Manual Review,The environment is good but the basic is too low.
3,Positive,3,0.357888,Manual Review,very traditional and corporate work culture
4,Negative,3,0.368872,Manual Review,Depends on department


Final_Label
Positive    744
Negative    448
3           115
2            84
Neutral      12
1            12
Name: count, dtype: int64


## Standardise Final Labels After Manual Sentiment Label Editing

To make manual checking faster and easier, numeric codes are used when editing the `Final_Label` column.

Use the following codes when correcting the AI-generated sentiment label:

| Code | Sentiment |
| ---- | --------- |
| `1`  | Positive  |
| `2`  | Negative  |
| `3`  | Neutral   |

For example, if the AI predicts **Positive** but the sentence is actually **Negative**, replace the value in `Final_Label` with `2`.

During manual review:

* Check whether the **sentence split is correct** and edit the `sentence_text` if necessary.
* Check whether the **AI sentiment label is correct**.
* If the label needs correction, enter `1`, `2`, or `3` in `Final_Label`.
* Leave the original `AI_Label` unchanged so that the AI prediction can still be compared with the manually corrected label.

After manual checking, the numeric codes will be converted back to their sentiment labels:

`1 → Positive`
`2 → Negative`
`3 → Neutral`


In [28]:
label_map = {
    1: "Positive",
    2: "Negative",
    3: "Neutral",
    "1": "Positive",
    "2": "Negative",
    "3": "Neutral"
}

checked_df["Final_Label"] = checked_df["Final_Label"].apply(
    lambda x: label_map.get(x, x)
)

print(checked_df["Final_Label"].value_counts(dropna=False))

Final_Label
Positive    756
Negative    532
Neutral     127
Name: count, dtype: int64


## Build final labelled text dataset

The final modelling task in the project is binary sentiment classification, so Neutral rows are removed after manual checking.

In [29]:
labelled_df = checked_df.copy()

drop_cols = [
    c for c in ["AI_Label", "AI_Confidence", "Review_Status"]
    if c in labelled_df.columns
]
labelled_df = labelled_df.drop(columns=drop_cols)

labelled_df = labelled_df.rename(columns={
    "Final_Label": "sentiment",
    "sentence_text": "review_text",
    "Review Text": "review_text"
})

required_cols = ["review_text", "sentiment"]
labelled_df = labelled_df.dropna(subset=required_cols).reset_index(drop=True)

print("Before removing Neutral:", len(labelled_df))

labelled_df = labelled_df[
    labelled_df["sentiment"].isin(["Positive", "Negative"])
].reset_index(drop=True)

print("After removing Neutral:", len(labelled_df))
print(labelled_df["sentiment"].value_counts())

display(labelled_df.head())

Before removing Neutral: 1415
After removing Neutral: 1288
sentiment
Positive    756
Negative    532
Name: count, dtype: int64


,sentiment,review_text
0,Negative,The environment is good but the basic is too low.
1,Negative,Still a lot of manual work and very volume bas...
2,Negative,Only give short break
3,Negative,but Depends on Your Tolerance with the Culture
4,Negative,Everything is manual and use a lot of paper.


## Save raw labelled sentence dataset

In [30]:
output_path = LABELED_DIR / "03_sentence_labeled_raw.csv"
labelled_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Saved:", output_path)

Saved: data/labeled/03_sentence_labeled_raw.csv
